# 🧠 GhostFaceNet → TFLite v3
### TF 2.19 + Python 3.12 + Keras 3.x Compatible

---

**Root cause of previous errors:**
- `AttributeError: keras has no attribute '__version__'` → TF 2.19 ships Keras 3.x which removed this attribute
- `OSError: file signature not found` → Google Drive downloaded HTML redirect, not real HDF5
- **Fix:** Use `tf_keras` (legacy Keras 2.x) for `.h5` loading + HuggingFace for downloads

| Step | Description |
|---|---|
| 1 | Install `tf_keras` (legacy) + `keras_cv_attention_models` |
| 2 | Imports using `tf_keras` — works with TF 2.19 |
| 3 | Config |
| 4 | Auto-download model (HuggingFace → DeepFace → gdown) |
| 5 | Load `.h5` via `tf_keras` + model surgery |
| 6 | Save to SavedModel (most stable TFLite conversion path) |
| 7 | Convert 4 TFLite variants |
| 8 | Verify + visualize |
| 9 | AMS integration test |
| 10 | Summary |

> ⚠️ **After Step 1 on Colab: Runtime → Restart session → Run from Step 2**

---
## 📦 Step 1 — Install Dependencies
> **After running this cell → Restart Colab runtime → then run from Step 2**

In [ ]:
# tf_keras = legacy Keras 2.x API — required to load GhostFaceNet .h5 files
# TF 2.19 ships Keras 3.x by default which breaks .h5 loading for custom models
!pip install -q tf_keras

# GhostFaceNet backbone + model surgery (grouped conv fix, GELU fix)
!pip install -q keras-cv-attention-models

# Download utilities
!pip install -q huggingface_hub gdown

# DeepFace (fallback downloader)
!pip install -q deepface

# Utils
!pip install -q Pillow matplotlib numpy scikit-learn tqdm

print("\n✅ Done — NOW RESTART YOUR RUNTIME then run from Step 2")
print("   Colab: Runtime → Restart session")

---
## 🔧 Step 2 — Imports (TF 2.19 Compatible)

In [ ]:
import os, sys, glob, random, shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# ── Critical: tell TF to use legacy Keras 2.x API ─────────────────────────────
# Must be set BEFORE importing tensorflow
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf

# Import tf_keras (legacy Keras 2.x) — NOT keras (which is now Keras 3.x)
# tf_keras supports .h5 model loading + model_surgery + TFLite conversion
import tf_keras as keras

# ── Version check ──────────────────────────────────────────────────────────────
print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"tf_keras   : {keras.__version__}")
print(f"NumPy      : {np.__version__}")

# ── GPU setup ─────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f"\n🚀 GPU : {gpus[0].name}")
else:
    print("\n⚠️  No GPU — CPU mode (conversion still works)")

---
## ⚙️ Step 3 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
WEIGHTS_DIR     = './weights'
OUTPUT_DIR      = './tflite_output'
SAVED_MODEL_DIR = './saved_model_tmp'

# ── Model selection ────────────────────────────────────────────────────────────
# Options:
#   'GhostFaceNet_W1.3_S1_ArcFace.h5'   99.78% LFW ← RECOMMENDED
#   'GhostFaceNet_W1.3_S2_ArcFace.h5'   99.71% LFW
#   'GhostFaceNet_W1.3_S1_CosFace.h5'   99.75% LFW
MODEL_FILENAME  = 'GhostFaceNet_W1.3_S1_ArcFace.h5'

# ── Input spec (standard for GhostFaceNet + ArcFace) ──────────────────────────
INPUT_H, INPUT_W, INPUT_C = 112, 112, 3
EMB_DIM = 512

# ── INT8 calibration images (optional, leave None for random noise) ────────────
CALIB_IMAGE_DIR = None   # '/content/face_images'
CALIB_COUNT     = 100

# ── Create directories ─────────────────────────────────────────────────────────
for d in [WEIGHTS_DIR, OUTPUT_DIR, SAVED_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

MODEL_H5_PATH = os.path.join(WEIGHTS_DIR, MODEL_FILENAME)

print(f"🎯 Model    : {MODEL_FILENAME}")
print(f"📐 Input    : ({INPUT_H}, {INPUT_W}, {INPUT_C})")
print(f"📊 Embedding: {EMB_DIM}-dim")
print(f"📁 Output   : {os.path.abspath(OUTPUT_DIR)}")

---
## ⬇️ Step 4 — Auto-Download GhostFaceNet `.h5`
3-method fallback chain. Validates HDF5 signature before and after download — no more HTML redirect corruption.

In [ ]:
HDF5_MAGIC = b'\x89HDF\r\n\x1a\n'

def is_valid_hdf5(path):
    try:
        with open(path, 'rb') as f:
            return f.read(8) == HDF5_MAGIC
    except Exception:
        return False


def try_huggingface(dest):
    print("  🔵 Trying HuggingFace Hub (serengil/deepface_models)...")
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(
            repo_id='serengil/deepface_models',
            filename=MODEL_FILENAME,
            local_dir=WEIGHTS_DIR,
        )
        if is_valid_hdf5(p):
            if p != dest:
                shutil.copy(p, dest)
            print(f"     ✅ Downloaded {os.path.getsize(dest)/(1024*1024):.1f} MB")
            return True
        print("     ❌ File corrupted (not HDF5)")
    except Exception as e:
        print(f"     ❌ Failed: {e}")
    return False


def try_deepface(dest):
    print("  🟡 Trying DeepFace built-in downloader...")
    try:
        df_cache = os.path.join(os.path.expanduser('~'), '.deepface', 'weights', MODEL_FILENAME)
        if is_valid_hdf5(df_cache):
            shutil.copy(df_cache, dest)
            print(f"     ✅ Cache hit — {os.path.getsize(dest)/(1024*1024):.1f} MB")
            return True
        from deepface import DeepFace
        dummy = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        Image.fromarray(dummy).save('/tmp/_dummy.png')
        try:
            DeepFace.represent('/tmp/_dummy.png', model_name='GhostFaceNet',
                               detector_backend='skip', enforce_detection=False)
        except Exception:
            pass
        if is_valid_hdf5(df_cache):
            shutil.copy(df_cache, dest)
            print(f"     ✅ Downloaded via DeepFace — {os.path.getsize(dest)/(1024*1024):.1f} MB")
            return True
        print("     ❌ DeepFace download did not produce valid HDF5")
    except Exception as e:
        print(f"     ❌ Failed: {e}")
    return False


def try_gdown(dest):
    print("  🟠 Trying gdown (Google Drive)...")
    # Known file IDs from: https://github.com/HamadYA/GhostFaceNets
    IDS = {
        'GhostFaceNet_W1.3_S1_ArcFace.h5': '1UoaMLae1h1B9B7a8lhXP6Cg4RKSz7s3n',
        'GhostFaceNet_W1.3_S2_ArcFace.h5': '1xM-KBiT_vIFiVMgLWOtBUxPD8EoZWfZd',
        'GhostFaceNet_W1.3_S1_CosFace.h5': '1x2tGpyIbFT79NjIMdaLhf8ukmAnrUXf5',
    }
    fid = IDS.get(MODEL_FILENAME)
    if not fid:
        print(f"     ❌ No Drive ID for {MODEL_FILENAME} — get it from GitHub README")
        return False
    try:
        import gdown
        gdown.download(f'https://drive.google.com/uc?id={fid}',
                       dest, fuzzy=True, quiet=False)
        if is_valid_hdf5(dest):
            print(f"     ✅ Downloaded {os.path.getsize(dest)/(1024*1024):.1f} MB")
            return True
        print("     ❌ Still HTML redirect — Drive quota hit, try HuggingFace manually")
    except Exception as e:
        print(f"     ❌ Failed: {e}")
    return False


# ── Main ──────────────────────────────────────────────────────────────────────
print(f"🔍 Checking: {MODEL_H5_PATH}")

if is_valid_hdf5(MODEL_H5_PATH):
    mb = os.path.getsize(MODEL_H5_PATH) / (1024*1024)
    print(f"✅ Valid HDF5 already exists ({mb:.1f} MB) — skipping download")
else:
    if os.path.exists(MODEL_H5_PATH):
        print("⚠️  Existing file is NOT valid HDF5 (HTML redirect?) — deleting and re-downloading")
        os.remove(MODEL_H5_PATH)
    print(f"\n⬇️  Downloading {MODEL_FILENAME}...\n")
    ok = try_huggingface(MODEL_H5_PATH) or try_deepface(MODEL_H5_PATH) or try_gdown(MODEL_H5_PATH)
    if not ok:
        raise RuntimeError(
            f"❌ All download methods failed!\n"
            f"Manual: go to https://github.com/HamadYA/GhostFaceNets\n"
            f"Download {MODEL_FILENAME} and place at {os.path.abspath(MODEL_H5_PATH)}"
        )

assert is_valid_hdf5(MODEL_H5_PATH), "❌ File is not valid HDF5 — delete and re-run"
print(f"\n🎉 Model ready: {os.path.abspath(MODEL_H5_PATH)} ({os.path.getsize(MODEL_H5_PATH)/(1024*1024):.1f} MB)")

---
## 🔬 Step 5 — Load Model + Model Surgery
Uses `tf_keras.models.load_model` (legacy Keras 2.x) — the only reliable way to load GhostFaceNet `.h5` on TF 2.19.

In [ ]:
# Must use tf_keras (Keras 2.x) — NOT keras.models.load_model (Keras 3.x)
# GhostFaceNet h5 files were saved with Keras 2.x and are incompatible with Keras 3.x
print("🔃 Loading model with tf_keras (legacy Keras 2.x)...")
model = keras.models.load_model(MODEL_H5_PATH, compile=False)

print(f"\n✅ Model loaded")
print(f"   Input  : {model.input_shape}")
print(f"   Output : {model.output_shape}")
print(f"   Params : {model.count_params():,}")

# Auto-update config if model shape differs from defaults
_, H, W, C = model.input_shape
D = model.output_shape[-1]
if (H, W) != (INPUT_H, INPUT_W):
    print(f"   ⚠️  Input shape updated: {INPUT_H}×{INPUT_W} → {H}×{W}")
    INPUT_H, INPUT_W = H, W
if D != EMB_DIM:
    print(f"   ⚠️  Embedding dim updated: {EMB_DIM} → {D}")
    EMB_DIM = D

# ── Model Surgery ─────────────────────────────────────────────────────────────
# Fixes 3 TFLite incompatibilities in GhostFaceNet:
#   1. Grouped Conv2D (groups > 1)  → split into standard Conv2D
#   2. GELU activation              → approximate GELU
#   3. tf.image.extract_patches     → Conv2D equivalent
print("\n🔧 Applying model surgery...")
try:
    from keras_cv_attention_models import model_surgery
    model_cvt = model_surgery.prepare_for_tflite(model)
    print("   ✅ prepare_for_tflite() applied (grouped conv + GELU + extract_patches fixed)")
except Exception as e:
    print(f"   ⚠️  model_surgery failed: {e}")
    print("   → Using original model with SELECT_TF_OPS fallback")
    model_cvt = model

# ── Post-surgery sanity check ─────────────────────────────────────────────────
np.random.seed(0)
dummy = np.random.uniform(-1, 1, (1, INPUT_H, INPUT_W, INPUT_C)).astype(np.float32)
out_orig = model.predict(dummy, verbose=0)
out_surg = model_cvt.predict(dummy, verbose=0)
diff = float(np.max(np.abs(out_orig - out_surg)))
print(f"\n🔬 Post-surgery max diff: {diff:.2e} {'✅ OK' if diff < 1e-4 else '⚠️  check surgery output'}")

---
## 💾 Step 6 — Save as SavedModel
SavedModel format is the most stable TFLite conversion path in TF 2.19. More reliable than converting directly from a Keras model object.

In [ ]:
print(f"💾 Saving to SavedModel format...")

# Clean previous SavedModel if exists
if os.path.exists(SAVED_MODEL_DIR):
    shutil.rmtree(SAVED_MODEL_DIR)

# Save using tf_keras (Keras 2.x) export
# Using tf.saved_model.save() with a concrete function for maximum compatibility
try:
    # Method 1: Direct SavedModel export (preferred)
    model_cvt.save(SAVED_MODEL_DIR, save_format='tf')
    print(f"   ✅ SavedModel saved → {SAVED_MODEL_DIR}")
except Exception as e:
    print(f"   ⚠️  Direct save failed: {e}")
    print("   → Trying concrete function export...")
    try:
        # Method 2: Via concrete function (more explicit)
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[None, INPUT_H, INPUT_W, INPUT_C],
                          dtype=tf.float32, name='input')
        ])
        def serving_fn(x):
            return model_cvt(x, training=False)

        tf.saved_model.save(
            model_cvt, SAVED_MODEL_DIR,
            signatures={'serving_default': serving_fn}
        )
        print(f"   ✅ SavedModel (concrete fn) saved → {SAVED_MODEL_DIR}")
    except Exception as e2:
        print(f"   ❌ Both save methods failed: {e2}")
        raise

# Verify SavedModel is readable
loaded = tf.saved_model.load(SAVED_MODEL_DIR)
print(f"   ✅ SavedModel verified — ready for TFLite conversion")

---
## 🛠️ Step 7 — Helpers

In [ ]:
def preprocess(img, size=None):
    """GhostFaceNet normalization: [0,255] uint8 → [-1,1] float32."""
    sz = size or (INPUT_H, INPUT_W)
    if isinstance(img, str):
        img = np.array(Image.open(img).convert('RGB').resize(sz), dtype=np.float32)
    else:
        img = np.asarray(img, dtype=np.float32)
    return np.expand_dims((img - 127.5) * 0.0078125, 0)  # (1,H,W,3)


def rep_dataset_gen():
    """Generator for INT8 full quantization calibration."""
    if CALIB_IMAGE_DIR and os.path.isdir(CALIB_IMAGE_DIR):
        paths = []
        for ext in ('*.jpg','*.jpeg','*.png','*.JPG'):
            paths += glob.glob(os.path.join(CALIB_IMAGE_DIR,'**',ext), recursive=True)
        random.shuffle(paths)
        print(f"   📸 Calibration: {min(len(paths),CALIB_COUNT)} real images")
        for p in paths[:CALIB_COUNT]:
            try: yield [preprocess(p)]
            except: pass
    else:
        print(f"   📸 Calibration: random noise ({CALIB_COUNT} samples)")
        for _ in range(CALIB_COUNT):
            yield [np.random.uniform(-1,1,(1,INPUT_H,INPUT_W,INPUT_C)).astype(np.float32)]


def run_tflite(path, inp):
    """Run inference on any TFLite model (float32/float16/int8)."""
    interp = tf.lite.Interpreter(model_path=path)
    interp.allocate_tensors()
    ind = interp.get_input_details()[0]
    outd = interp.get_output_details()[0]
    dtype = ind['dtype']
    if dtype in (np.int8, np.uint8):
        s, z = ind['quantization']
        data = np.clip(inp/s + z, np.iinfo(dtype).min, np.iinfo(dtype).max).astype(dtype)
    else:
        data = inp.astype(np.float32)
    interp.set_tensor(ind['index'], data)
    interp.invoke()
    out = interp.get_tensor(outd['index'])
    if outd['dtype'] in (np.int8, np.uint8):
        s, z = outd['quantization']
        out = (out.astype(np.float32) - z) * s
    return out


def cosine_sim(a, b):
    a, b = a.flatten(), b.flatten()
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-10))


def build_converter(quant_type, rep_fn=None):
    """Build TFLiteConverter from SavedModel — most compatible path in TF 2.19."""
    conv = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    conv.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    conv._experimental_lower_tensor_list_ops = False
    conv.experimental_new_converter = True  # MLIR-based converter (default TF 2.19)

    if quant_type == 'float32':
        pass

    elif quant_type == 'float16':
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic_int8':
        # Weights → INT8, activations dynamic at runtime
        # No calibration needed — best for old Android (API 21+)
        conv.optimizations = [tf.lite.Optimize.DEFAULT]

    elif quant_type == 'full_int8':
        # Weights + activations → INT8, enables Android NNAPI
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = rep_fn
        conv.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]
        # Keep IO float32 — easier React Native integration
        conv.inference_input_type  = tf.float32
        conv.inference_output_type = tf.float32

    return conv


print("✅ Helpers ready")

---
## 🔄 Step 8 — Convert All 4 TFLite Variants

In [ ]:
VARIANTS = [
    ('float32',      'ghostfacenet_float32.tflite',      None),
    ('float16',      'ghostfacenet_float16.tflite',      None),
    ('dynamic_int8', 'ghostfacenet_dynamic_int8.tflite', None),
    ('full_int8',    'ghostfacenet_full_int8.tflite',    rep_dataset_gen),
]

results = {}

for quant_type, fname, rep_fn in VARIANTS:
    out_path = os.path.join(OUTPUT_DIR, fname)
    print(f"\n{'─'*52}")
    print(f"🔄 [{quant_type}]")
    try:
        conv  = build_converter(quant_type, rep_fn)
        model_bytes = conv.convert()
        with open(out_path, 'wb') as f:
            f.write(model_bytes)
        mb = os.path.getsize(out_path) / (1024*1024)
        results[quant_type] = {'path': out_path, 'size': mb, 'ok': True}
        print(f"   ✅ {fname} ({mb:.2f} MB)")
    except Exception as e:
        results[quant_type] = {'path': out_path, 'size': 0, 'ok': False, 'err': str(e)}
        print(f"   ❌ Failed: {e}")

ok_count = sum(v['ok'] for v in results.values())
print(f"\n\n{'='*52}")
print(f"  {ok_count}/{len(VARIANTS)} variants converted successfully")
print(f"{'='*52}")

---
## ✅ Step 9 — Accuracy Verification

In [ ]:
np.random.seed(42)
test_raw  = np.random.randint(0, 255, (INPUT_H, INPUT_W, INPUT_C)).astype(np.float32)
test_norm = preprocess(test_raw)

# Ground truth from original Keras model
keras_emb = model.predict(test_norm, verbose=0)
print(f"Keras  — shape: {keras_emb.shape} | norm: {np.linalg.norm(keras_emb):.4f}\n")

verify = []
print(f"{'Variant':<22} {'Cosine Sim':>11} {'L2 Diff':>9} {'Size MB':>9}  Status")
print('─' * 62)

for qt, info in results.items():
    if not info['ok']:
        print(f"{qt:<22} {'SKIPPED':>43}")
        continue
    try:
        tfl_emb = run_tflite(info['path'], test_norm)
        sim = cosine_sim(keras_emb, tfl_emb)
        l2  = float(np.linalg.norm(keras_emb - tfl_emb))
        flag = '🟢' if sim > 0.999 else ('🟡' if sim > 0.990 else '🔴')
        print(f"{qt:<22} {sim:>11.6f} {l2:>9.5f} {info['size']:>9.2f}  {flag}")
        verify.append({'name': qt, 'sim': sim, 'size': info['size']})
    except Exception as e:
        print(f"{qt:<22} ERROR: {e}")

print('─' * 62)
print("🟢 >0.999 excellent  |  🟡 >0.990 acceptable  |  🔴 review")

---
## 📊 Step 10 — Visualize

In [ ]:
if verify:
    names  = [r['name'] for r in verify]
    sims   = [r['sim']  for r in verify]
    sizes  = [r['size'] for r in verify]
    cols   = ['#10B981' if s>.999 else '#F59E0B' if s>.990 else '#EF4444' for s in sims]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#F8F7FF')
    fig.suptitle('GhostFaceNet → TFLite  |  TF 2.19 + Python 3.12',
                 fontsize=13, fontweight='bold', color='#1E1B4B')

    for ax, vals, col_list, title, ylabel in [
        (ax1, sims,  cols,             'Accuracy (Cosine Sim vs Keras)', 'Cosine Similarity'),
        (ax2, sizes, ['#4F46E5']*len(names), 'Model Size',                    'MB'),
    ]:
        ax.set_facecolor('#F8F7FF')
        bars = ax.bar(names, vals, color=col_list, edgecolor='white', lw=1.5, zorder=3)
        ax.set_title(title, fontweight='bold', color='#1E1B4B')
        ax.set_ylabel(ylabel, color='#1E1B4B')
        ax.grid(axis='y', alpha=0.3, zorder=0)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2,
                    v + (0.0003 if ax is ax1 else 0.1),
                    f'{v:.5f}' if ax is ax1 else f'{v:.1f}MB',
                    ha='center', va='bottom', fontsize=8, fontweight='bold', color='#1E1B4B')

    ax1.set_ylim([min(sims)-.005, 1.001])
    ax1.axhline(0.999, color='#10B981', ls='--', alpha=.6, label='Excellent')
    ax1.axhline(0.990, color='#F59E0B', ls='--', alpha=.6, label='Acceptable')
    ax1.legend(fontsize=8)

    plt.tight_layout()
    out_png = os.path.join(OUTPUT_DIR, 'results.png')
    plt.savefig(out_png, dpi=150, bbox_inches='tight', facecolor='#F8F7FF')
    plt.show()
    print(f"📊 Saved → {out_png}")

---
## 📱 Step 11 — AMS Integration Test
Simulates enrollment (5 embeddings → centroid) + recognition (max-sim voting + 3-frame guard).

In [ ]:
PRIORITY = ['dynamic_int8', 'full_int8', 'float16', 'float32']
mdl_path = next((results[q]['path'] for q in PRIORITY if results.get(q,{}).get('ok')), None)
if not mdl_path:
    raise RuntimeError("No successful TFLite model to test")

print(f"🏢 AMS Integration Test  |  {os.path.basename(mdl_path)}\n")

THRESHOLD          = 0.60
N_EMP              = 3
EMB_PER_EMP        = 5
CONFIRM_FRAMES     = 3
np.random.seed(7)

# ── Enrollment ────────────────────────────────────────────────────────────────
print("📋 Enrolling...")
db = {}
for i in range(N_EMP):
    base = np.random.uniform(0, 255, (INPUT_H, INPUT_W, INPUT_C)).astype(np.float32)
    embs = []
    for _ in range(EMB_PER_EMP):
        noisy = np.clip(base + np.random.normal(0, 5, base.shape), 0, 255)
        e = run_tflite(mdl_path, preprocess(noisy)).flatten()
        e /= np.linalg.norm(e) + 1e-10
        embs.append(e)
    centroid = np.mean(embs, axis=0)
    centroid /= np.linalg.norm(centroid) + 1e-10
    db[f'emp_{i:03d}'] = {'embs': embs, 'centroid': centroid, 'base': base}
    print(f"   ✅ emp_{i:03d} — {EMB_PER_EMP} embeddings enrolled")

# ── Recognition ───────────────────────────────────────────────────────────────
print("\n🔍 Recognition (max-sim voting + 3-frame confirmation)...\n")
all_ok = True
for tid, data in db.items():
    hits = 0
    last_score, last_match = 0, ''
    for _ in range(CONFIRM_FRAMES):
        live = np.clip(data['base'] + np.random.normal(0, 8, data['base'].shape), 0, 255)
        le   = run_tflite(mdl_path, preprocess(live)).flatten()
        le  /= np.linalg.norm(le) + 1e-10
        scores = {eid: max(cosine_sim(le, e) for e in d['embs']) for eid, d in db.items()}
        best   = max(scores, key=scores.get)
        score  = scores[best]
        last_score, last_match = score, best
        if best == tid and score >= THRESHOLD:
            hits += 1
    ok = hits == CONFIRM_FRAMES
    all_ok = all_ok and ok
    print(f"   {'✅' if ok else '❌'} {tid} → {last_match}  "
          f"score={last_score:.4f}  confirmed={hits}/{CONFIRM_FRAMES}")

print(f"\n{'🎉 All employees recognised!' if all_ok else '⚠️  Some failed — check threshold'}")

---
## 📋 Step 12 — Final Summary

In [ ]:
print("\n" + "═"*64)
print("  ✅  GHOSTFACENET → TFLITE COMPLETE  (TF 2.19 / Python 3.12)")
print("═"*64)

INFO = {
    'float32':      ('Float32  (baseline)',  'High-end Android API 28+'),
    'float16':      ('Float16  (half)',      'Mid-range Android API 26+'),
    'dynamic_int8': ('Dynamic INT8',         '✅  RECOMMENDED — Any Android API 21+'),
    'full_int8':    ('Full INT8  + NNAPI',   'Best perf on old Snapdragon/MediaTek'),
}
for qt, (label, note) in INFO.items():
    info = results.get(qt, {})
    if info.get('ok'):
        print(f"  {label:25s}  {info['size']:5.1f} MB  →  {note}")
    else:
        print(f"  {label:25s}  FAILED")

print("═"*64)
print()
print("📱 DROP INTO YOUR AMS APP:")
print("   android/app/src/main/assets/ghostfacenet.tflite")
print("   (use dynamic_int8 variant)")
print()
print("🔧 UPDATE config.ts:")
print(f"   inputSize    : 192  →  {INPUT_H}    // update resize plugin call")
print(f"   embeddingDim : 192  →  {EMB_DIM}   // update cosine sim arrays")
print(f"   normalize    : same →  (pixel - 127.5) * 0.0078125")
print(f"   modelFile    : 'ghostfacenet.tflite'")
print(f"   runSync API  : same →  model.runSync([input])")
print()
print(f"📂 Output dir: {os.path.abspath(OUTPUT_DIR)}")
print("═"*64)